In [24]:
import sqlite3
import pandas as pd

# Create an in-memory SQLite database
conn = sqlite3.connect(":memory:")

# Cursor
cursor = conn.cursor()

print("SQLite database connected successfully!")

SQLite database connected successfully!


In [25]:
cursor.executescript("""
CREATE TABLE customers (
    customer_id INTEGER PRIMARY KEY,
    name TEXT,
    city TEXT,
    signup_date DATE
);

CREATE TABLE orders (
    order_id INTEGER PRIMARY KEY,
    customer_id INTEGER,
    order_date DATE,
    amount REAL,
    status TEXT,
    FOREIGN KEY (customer_id) REFERENCES customers(customer_id)
);

CREATE TABLE products (
    product_id INTEGER PRIMARY KEY,
    product_name TEXT,
    category TEXT,
    price REAL
);

CREATE TABLE order_items (
    item_id INTEGER PRIMARY KEY,
    order_id INTEGER,
    product_id INTEGER,
    quantity INTEGER,
    FOREIGN KEY (order_id) REFERENCES orders(order_id),
    FOREIGN KEY (product_id) REFERENCES products(product_id)
);
""")

print("Tables created!")

Tables created!


In [26]:
cursor.executescript("""
INSERT INTO customers VALUES
(1, 'Alice', 'New York', '2023-01-15'),
(2, 'Bob', 'London', '2023-02-20'),
(3, 'Charlie', 'Paris', '2023-03-10'),
(4, 'Diana', 'New York', '2023-04-05'),
(5, 'Eve', 'Berlin', '2023-05-12'),
(6, 'Frank', 'London', '2023-06-01');

INSERT INTO orders VALUES
(101, 1, '2024-01-10', 250.00, 'completed'),
(102, 1, '2024-02-15', 120.00, 'completed'),
(103, 2, '2024-01-20', 500.00, 'completed'),
(104, 3, '2024-03-05', 300.00, 'cancelled'),
(105, 3, '2024-03-10', 450.00, 'completed'),
(106, 4, '2024-02-01', 200.00, 'completed'),
(107, 5, '2024-04-12', 150.00, 'pending'),
(108, 5, '2024-04-20', 600.00, 'completed'),
(109, 1, '2024-05-01', 800.00, 'completed'),
(110, 2, '2024-05-15', 350.00, 'completed');

INSERT INTO products VALUES
(1, 'Laptop', 'Electronics', 1200.00),
(2, 'Phone', 'Electronics', 800.00),
(3, 'Tablet', 'Electronics', 450.00),
(4, 'Chair', 'Furniture', 200.00),
(5, 'Desk', 'Furniture', 350.00),
(6, 'Notebook', 'Stationery', 5.00);

INSERT INTO order_items VALUES
(1, 101, 4, 1),
(2, 101, 6, 10),
(3, 102, 6, 24),
(4, 103, 2, 1),
(5, 104, 3, 1),
(6, 105, 1, 1),
(7, 106, 4, 1),
(8, 107, 6, 30),
(9, 108, 1, 1),
(10, 109, 2, 1),
(11, 110, 5, 1);
""")

conn.commit()

print("Data inserted successfully!")

Data inserted successfully!


In [27]:
def run_sql(query):
    return pd.read_sql_query(query, conn)

In [28]:
run_sql("SELECT * FROM customers;")

,customer_id,name,city,signup_date
0,1,Alice,New York,2023-01-15
1,2,Bob,London,2023-02-20
2,3,Charlie,Paris,2023-03-10
3,4,Diana,New York,2023-04-05
4,5,Eve,Berlin,2023-05-12
5,6,Frank,London,2023-06-01


In [29]:
run_sql("SELECT * FROM orders;")

,order_id,customer_id,order_date,amount,status
0,101,1,2024-01-10,250.0,completed
1,102,1,2024-02-15,120.0,completed
2,103,2,2024-01-20,500.0,completed
3,104,3,2024-03-05,300.0,cancelled
4,105,3,2024-03-10,450.0,completed
5,106,4,2024-02-01,200.0,completed
6,107,5,2024-04-12,150.0,pending
7,108,5,2024-04-20,600.0,completed
8,109,1,2024-05-01,800.0,completed
9,110,2,2024-05-15,350.0,completed


In [30]:
run_sql("SELECT * FROM products;")

,product_id,product_name,category,price
0,1,Laptop,Electronics,1200.0
1,2,Phone,Electronics,800.0
2,3,Tablet,Electronics,450.0
3,4,Chair,Furniture,200.0
4,5,Desk,Furniture,350.0
5,6,Notebook,Stationery,5.0


In [58]:
run_sql("SELECT * FROM order_items;")

,item_id,order_id,product_id,quantity
0,1,101,4,1
1,2,101,6,10
2,3,102,6,24
3,4,103,2,1
4,5,104,3,1
5,6,105,1,1
6,7,106,4,1
7,8,107,6,30
8,9,108,1,1
9,10,109,2,1


-- ============================================================
-- Q1: Find the total revenue from completed orders
-- ============================================================
-- Reasoning: Filter to completed, then SUM the amount.


In [61]:
run_sql("""
-- ============================================================
-- Q1: Find the total revenue from completed orders
-- ============================================================
-- Reasoning: Filter to completed orders, then SUM the amount.

SELECT SUM(amount) AS total_revenue
FROM orders
WHERE status = 'completed';
""")

,total_revenue
0,3270.0


In [62]:
run_sql("""
-- ============================================================
-- Q2: List all customers from New York or London,
--     sorted by signup date
-- ============================================================
-- Reasoning: Filter customers from New York or London,
-- then sort them by signup date.

SELECT name, city, signup_date
FROM customers
WHERE city IN ('New York', 'London')
ORDER BY signup_date;
""")

,name,city,signup_date
0,Alice,New York,2023-01-15
1,Bob,London,2023-02-20
2,Diana,New York,2023-04-05
3,Frank,London,2023-06-01


In [63]:
run_sql("""
-- ============================================================
-- Q3: How many orders were placed in each status?
-- ============================================================
-- Reasoning: Group orders by status and count the orders
-- in each group.

SELECT status,
       COUNT(*) AS order_count
FROM orders
GROUP BY status;
""")

,status,order_count
0,cancelled,1
1,completed,8
2,pending,1


In [64]:
run_sql("""
-- ============================================================
-- Q4: Find the average order amount per city
-- ============================================================
-- Reasoning: Join orders with customers to get the city,
-- then calculate the average order amount for each city.

SELECT c.city,
       AVG(o.amount) AS avg_order_amount
FROM orders o
JOIN customers c
    ON o.customer_id = c.customer_id
GROUP BY c.city;
""")

,city,avg_order_amount
0,Berlin,375.0
1,London,425.0
2,New York,342.5
3,Paris,375.0


In [65]:
run_sql("""
-- ============================================================
-- Q5: List customers who have never placed an order
-- ============================================================
-- Reasoning: LEFT JOIN customers with orders.
-- Customers without orders will have NULL order_id.

SELECT c.name
FROM customers c
LEFT JOIN orders o
    ON c.customer_id = o.customer_id
WHERE o.order_id IS NULL;
""")

,name
0,Frank


In [66]:
run_sql("""
-- ============================================================
-- Q6: Find the top 3 customers by total completed order amount
-- ============================================================
-- Reasoning: Filter completed orders, group by customer,
-- calculate total amount, sort descending, and take 3.

SELECT c.name,
       SUM(o.amount) AS total_amount
FROM orders o
JOIN customers c
    ON o.customer_id = c.customer_id
WHERE o.status = 'completed'
GROUP BY c.name
ORDER BY total_amount DESC
LIMIT 3;
""")

,name,total_amount
0,Alice,1170.0
1,Bob,850.0
2,Eve,600.0


In [67]:
run_sql("""
-- ============================================================
-- Q7: For each customer, show their total order amount
--     and the number of orders
-- ============================================================
-- Reasoning: LEFT JOIN keeps customers even if they have
-- no orders. SUM gives total amount and COUNT gives
-- the number of orders.

SELECT c.name,
       SUM(o.amount) AS total_amount,
       COUNT(o.order_id) AS order_count
FROM customers c
LEFT JOIN orders o
    ON c.customer_id = o.customer_id
GROUP BY c.name;
""")

,name,total_amount,order_count
0,Alice,1170.0,3
1,Bob,850.0,2
2,Charlie,750.0,2
3,Diana,200.0,1
4,Eve,750.0,2
5,Frank,NaN,0


In [68]:
run_sql("""
-- ============================================================
-- Q8: Find customers who have placed more than 2 orders
-- ============================================================
-- Reasoning: Group orders by customer, count their orders,
-- then use HAVING to keep counts greater than 2.

SELECT c.name,
       COUNT(o.order_id) AS order_count
FROM customers c
JOIN orders o
    ON c.customer_id = o.customer_id
GROUP BY c.name
HAVING COUNT(o.order_id) > 2;
""")

,name,order_count
0,Alice,3


In [69]:
run_sql("""
-- ============================================================
-- Q9: For each month of 2024, show the total revenue
--     from completed orders
-- ============================================================
-- Reasoning: Filter completed orders from 2024, extract
-- year-month, group by month, and calculate revenue.

SELECT strftime('%Y-%m', order_date) AS month,
       SUM(amount) AS revenue
FROM orders
WHERE status = 'completed'
  AND strftime('%Y', order_date) = '2024'
GROUP BY month
ORDER BY month;
""")

,month,revenue
0,2024-01,750.0
1,2024-02,320.0
2,2024-03,450.0
3,2024-04,600.0
4,2024-05,1150.0


In [70]:
run_sql("""
-- ============================================================
-- Q10: List the top-selling product by total quantity
--      in the Electronics category
-- ============================================================
-- Reasoning: Join order_items with products, filter Electronics,
-- sum quantity for each product, then take the highest.

SELECT p.product_name,
       SUM(oi.quantity) AS total_qty
FROM order_items oi
JOIN products p
    ON oi.product_id = p.product_id
WHERE p.category = 'Electronics'
GROUP BY p.product_name
ORDER BY total_qty DESC
LIMIT 1;
""")

,product_name,total_qty
0,Phone,2


In [71]:
run_sql("""
-- ============================================================
-- Q11: Rank customers by total completed order amount
--      using RANK()
-- ============================================================
-- Reasoning: First calculate each customer's total completed
-- amount using a CTE. Then rank those totals from highest
-- to lowest.

WITH totals AS (
    SELECT c.customer_id,
           c.name,
           SUM(o.amount) AS total_amount
    FROM orders o
    JOIN customers c
        ON o.customer_id = c.customer_id
    WHERE o.status = 'completed'
    GROUP BY c.customer_id, c.name
)

SELECT name,
       total_amount,
       RANK() OVER (
           ORDER BY total_amount DESC
       ) AS rnk
FROM totals;
""")

,name,total_amount,rnk
0,Alice,1170.0,1
1,Bob,850.0,2
2,Eve,600.0,3
3,Charlie,450.0,4
4,Diana,200.0,5


In [72]:
run_sql("""
-- ============================================================
-- Q12: For each customer, show their orders with a running
--      total of amount
-- ============================================================
-- Reasoning: PARTITION BY creates a separate running total
-- for each customer. ORDER BY makes it accumulate by date.

SELECT c.name,
       o.order_date,
       o.amount,
       SUM(o.amount) OVER (
           PARTITION BY o.customer_id
           ORDER BY o.order_date
       ) AS running_total
FROM orders o
JOIN customers c
    ON o.customer_id = c.customer_id
ORDER BY c.name, o.order_date;
""")

,name,order_date,amount,running_total
0,Alice,2024-01-10,250.0,250.0
1,Alice,2024-02-15,120.0,370.0
2,Alice,2024-05-01,800.0,1170.0
3,Bob,2024-01-20,500.0,500.0
4,Bob,2024-05-15,350.0,850.0
5,Charlie,2024-03-05,300.0,300.0
6,Charlie,2024-03-10,450.0,750.0
7,Diana,2024-02-01,200.0,200.0
8,Eve,2024-04-12,150.0,150.0
9,Eve,2024-04-20,600.0,750.0


In [73]:
run_sql("""
-- ============================================================
-- Q12: For each customer, show their orders with a running
--      total of amount
-- ============================================================
-- Reasoning: PARTITION BY creates a separate running total
-- for each customer. ORDER BY makes it accumulate by date.

SELECT c.name,
       o.order_date,
       o.amount,
       SUM(o.amount) OVER (
           PARTITION BY o.customer_id
           ORDER BY o.order_date
       ) AS running_total
FROM orders o
JOIN customers c
    ON o.customer_id = c.customer_id
ORDER BY c.name, o.order_date;
""")

,name,order_date,amount,running_total
0,Alice,2024-01-10,250.0,250.0
1,Alice,2024-02-15,120.0,370.0
2,Alice,2024-05-01,800.0,1170.0
3,Bob,2024-01-20,500.0,500.0
4,Bob,2024-05-15,350.0,850.0
5,Charlie,2024-03-05,300.0,300.0
6,Charlie,2024-03-10,450.0,750.0
7,Diana,2024-02-01,200.0,200.0
8,Eve,2024-04-12,150.0,150.0
9,Eve,2024-04-20,600.0,750.0
